# Comprehensive Code Generation Analysis

**Purpose**: Unified analysis of ALL code generation experiments across Phase 3.

**Dataset**: HumanEval (164 Python programming problems)

**Experiments**: 8 experiments across mars_codegen/ and runpod_codegen/

**Research Questions**:
- **RQ1**: How do model size and reasoning capabilities affect code generation accuracy (Pass@1)?
- **RQ2**: What is the trade-off between prompting strategy and performance/energy?
- **RQ3**: How does hardware (RTX A5000 vs H100) impact performance and energy efficiency?
- **RQ4**: How does code generation compare to vulnerability detection in terms of performance and energy?

**Date**: November 9, 2025

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

# Paths
PROJECT_ROOT = Path.cwd().parent
CODEGEN_DATA = PROJECT_ROOT / 'results' / 'analysis' / 'code_generation_master_dataset.csv'
VULN_DATA = PROJECT_ROOT / 'results' / 'analysis' / 'vuln_detection_master_dataset.csv'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Project Root: {PROJECT_ROOT}")
print(f"📊 Code Gen Data: {CODEGEN_DATA}")
print(f"📊 Vuln Detection Data: {VULN_DATA}")
print(f"💾 Output Directory: {OUTPUT_DIR}")

In [ ]:
# Load code generation data
df_codegen = pd.read_csv(CODEGEN_DATA)

print(f"\n📊 Code Generation Dataset loaded: {len(df_codegen)} experiments")
print(f"\nColumns: {list(df_codegen.columns)}")

# Display first few rows
df_codegen.head()

## 2. Data Cleaning & Preprocessing

In [ ]:
# Remove experiments with missing energy data
print("🧹 Cleaning data...")
print(f"Original size: {len(df_codegen)} experiments")

df_clean = df_codegen.dropna(subset=['energy_consumed_kwh']).copy()

print(f"After removing experiments with missing energy data: {len(df_clean)} experiments")
print(f"Removed: {len(df_codegen) - len(df_clean)} experiments\n")

# Duration in hours
df_clean['duration_hours'] = df_clean['duration_seconds'] / 3600

# Energy per sample
df_clean['energy_per_sample_kwh'] = df_clean['energy_consumed_kwh'] / df_clean['total_samples']
df_clean['emissions_per_sample_kg'] = df_clean['emissions_kg_codecarbon'] / df_clean['total_samples']

print("✅ Data cleaning complete!")
df_clean.head()

## 3. Exploratory Data Analysis

In [ ]:
# Summary statistics
print("📈 Summary Statistics\n")
print("="*80)

summary_cols = ['pass_at_1', 'pass_rate_pct', 'energy_consumed_kwh', 
                'emissions_kg_codecarbon', 'duration_hours']

print(df_clean[summary_cols].describe().round(3))

# Count by categories
print("\n" + "="*80)
print("\n📊 Experiment Breakdown:\n")
print(f"By Phase:\n{df_clean['phase'].value_counts()}\n")
print(f"By Model Size:\n{df_clean['model_size'].value_counts()}\n")
print(f"By Model Type:\n{df_clean['model_type'].value_counts()}\n")
print(f"By Prompting:\n{df_clean['prompting'].value_counts()}\n")
print(f"By Hardware:\n{df_clean['hardware'].value_counts()}")

# Missing data analysis
if len(df_codegen) > len(df_clean):
    missing = df_codegen[df_codegen['energy_consumed_kwh'].isna()]
    print("\n⚠️  Experiments with missing energy data:")
    for _, row in missing.iterrows():
        print(f"   - {row['experiment_id']}")

## 4. RQ1: Model Size & Reasoning Impact on Code Generation

### 4.1 Performance by Model Size (4B vs 30B)

In [ ]:
# Group by model size
model_size_perf = df_clean.groupby('model_size')[['pass_at_1', 'pass_rate_pct']].mean()

print("📊 Pass@1 Performance by Model Size:\n")
print(model_size_perf.round(2))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
model_size_perf['pass_rate_pct'].plot(kind='bar', ax=axes[0], rot=0, color=['skyblue', 'coral'])
axes[0].set_title('Pass@1 Performance by Model Size', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Model Size', fontsize=12)
axes[0].set_ylabel('Pass@1 (%)', fontsize=12)
axes[0].set_ylim(85, 100)
axes[0].grid(True, alpha=0.3)

# Add value labels
for i, v in enumerate(model_size_perf['pass_rate_pct']):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

# Box plot
df_clean.boxplot(column='pass_rate_pct', by='model_size', ax=axes[1])
axes[1].set_title('Pass@1 Distribution by Model Size', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Model Size', fontsize=12)
axes[1].set_ylabel('Pass@1 (%)', fontsize=12)
plt.suptitle('')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'codegen_model_size_performance.png', dpi=300, bbox_inches='tight')
plt.show()

# Statistical test
from scipy import stats
pass1_4b = df_clean[df_clean['model_size'] == '4B']['pass_at_1']
pass1_30b = df_clean[df_clean['model_size'] == '30B']['pass_at_1']
t_stat, p_value = stats.ttest_ind(pass1_4b, pass1_30b, nan_policy='omit')
print(f"\n📊 T-test (4B vs 30B Pass@1): t={t_stat:.3f}, p={p_value:.3f}")
if p_value < 0.05:
    print("   ✅ Statistically significant difference!")
else:
    print("   ⚠️  No statistically significant difference")

### 4.2 Performance by Model Type (Instruct vs Thinking)

In [ ]:
# Group by model type
model_type_perf = df_clean.groupby('model_type')[['pass_at_1', 'pass_rate_pct']].mean()

print("📊 Pass@1 Performance by Model Type:\n")
print(model_type_perf.round(2))

# Visualization
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
model_type_perf['pass_rate_pct'].plot(kind='bar', ax=ax, rot=0, color=['#FF6B6B', '#4ECDC4'])
ax.set_title('Pass@1 Performance by Model Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Model Type', fontsize=12)
ax.set_ylabel('Pass@1 (%)', fontsize=12)
ax.set_ylim(85, 100)
ax.grid(True, alpha=0.3)

# Add value labels
for i, v in enumerate(model_type_perf['pass_rate_pct']):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'codegen_model_type_performance.png', dpi=300, bbox_inches='tight')
plt.show()

# Statistical test
pass1_instruct = df_clean[df_clean['model_type'] == 'Instruct']['pass_at_1']
pass1_thinking = df_clean[df_clean['model_type'] == 'Thinking']['pass_at_1']
t_stat, p_value = stats.ttest_ind(pass1_instruct, pass1_thinking, nan_policy='omit')
print(f"\n📊 T-test (Instruct vs Thinking Pass@1): t={t_stat:.3f}, p={p_value:.3f}")
if p_value < 0.05:
    print("   ✅ Statistically significant difference!")
else:
    print("   ⚠️  No statistically significant difference")

### 4.3 Combined: Model Size × Model Type Heatmap

In [ ]:
# Pivot table for heatmap
heatmap_data = df_clean.pivot_table(
    values='pass_rate_pct',
    index='model_size',
    columns='model_type',
    aggfunc='mean'
)

print("📊 Pass@1 Heatmap (Model Size × Model Type):\n")
print(heatmap_data.round(2))

# Visualization
plt.figure(figsize=(8, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdYlGn', vmin=85, vmax=100, 
            cbar_kws={'label': 'Pass@1 (%)'})
plt.title('Code Generation Pass@1: Model Size × Model Type', fontsize=14, fontweight='bold')
plt.xlabel('Model Type', fontsize=12)
plt.ylabel('Model Size', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'codegen_pass1_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. RQ2: Prompting Strategy Analysis

In [ ]:
# Group by prompting strategy
prompting_perf = df_clean.groupby('prompting')[['pass_at_1', 'pass_rate_pct']].mean()

print("📊 Pass@1 Performance by Prompting Strategy:\n")
print(prompting_perf.round(2))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
prompting_perf['pass_rate_pct'].plot(kind='bar', ax=axes[0], rot=0, color=['#3498DB', '#E74C3C'])
axes[0].set_title('Pass@1 by Prompting Strategy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Prompting', fontsize=12)
axes[0].set_ylabel('Pass@1 (%)', fontsize=12)
axes[0].set_ylim(85, 100)
axes[0].grid(True, alpha=0.3)

# Add value labels
for i, v in enumerate(prompting_perf['pass_rate_pct']):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

# Box plot
df_clean.boxplot(column='pass_rate_pct', by='prompting', ax=axes[1])
axes[1].set_title('Pass@1 Distribution by Prompting', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Prompting', fontsize=12)
axes[1].set_ylabel('Pass@1 (%)', fontsize=12)
plt.suptitle('')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'codegen_prompting_performance.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. RQ3: Hardware Comparison

In [ ]:
### 7.3 Enhanced Pass@1 vs Energy Tradeoff (with Labels)

# Create comprehensive labeled scatter plot
fig, ax = plt.subplots(1, 1, figsize=(16, 10))

# Define colors and markers
size_colors = {'4B': '#3498DB', '30B': '#9B59B6'}  # Blue for 4B, Purple for 30B
type_markers = {'Instruct': 'o', 'Thinking': 's'}  # Circle for Instruct, Square for Thinking
hardware_edge_colors = {'Mars (RTX A5000)': '#2C3E50', 'RunPod (H100)': '#E67E22'}  # Dark gray for Mars, Orange for H100

# Plot each experiment with labels
for idx, row in df_clean.iterrows():
    model_size = row['model_size']
    model_type = row['model_type']
    prompting = row['prompting']
    hardware = row['hardware']

    # All code generation experiments use same approach, so all filled
    facecolor = size_colors[model_size]
    
    # Edge color based on hardware
    edgecolor = hardware_edge_colors[hardware]
    linewidth = 2.5

    # Plot the point
    ax.scatter(
        row['energy_consumed_kwh'],
        row['pass_rate_pct'],
        s=250,
        marker=type_markers[model_type],
        facecolors=facecolor,
        edgecolors=edgecolor,
        linewidths=linewidth,
        alpha=0.8,
        zorder=3
    )

    # Create label: "30B Inst F" format
    size_label = model_size
    type_label = 'Inst' if model_type == 'Instruct' else 'Thin'
    prompt_label = prompting[0].upper()  # Z for Zero-shot, F for Few-shot

    label = f"{size_label} {type_label} {prompt_label}"

    # Add label with offset
    ax.annotate(
        label,
        xy=(row['energy_consumed_kwh'], row['pass_rate_pct']),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.7)
    )

# Add trend line
from scipy import stats as sp_stats
x = df_clean['energy_consumed_kwh'].values
y = df_clean['pass_rate_pct'].values
slope, intercept, r_value, p_value, std_err = sp_stats.linregress(x, y)
line_x = np.array([x.min(), x.max()])
line_y = slope * line_x + intercept
ax.plot(line_x, line_y, '--', color='gray', linewidth=2, alpha=0.5,
        label=f'Trend (R²={r_value**2:.3f})', zorder=1)

# Styling
ax.set_xlabel('Energy Consumption (kWh)', fontsize=14, fontweight='bold')
ax.set_ylabel('Pass@1 (%)', fontsize=14, fontweight='bold')
ax.set_title('Code Generation Energy-Performance Tradeoff: All Experiments\n' +
             '4B & 30B | Instruct & Thinking | Zero-shot & Few-shot | Mars & H100',
             fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, zorder=0)
ax.set_ylim(85, 102)

# Create custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498DB', markeredgecolor='gray',
           markersize=10, markeredgewidth=2.5, label='4B Models', linewidth=0),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#9B59B6', markeredgecolor='gray',
           markersize=10, markeredgewidth=2.5, label='30B Models', linewidth=0),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markeredgecolor='gray',
           markersize=10, markeredgewidth=2.5, label='Instruct', linewidth=0),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', markeredgecolor='gray',
           markersize=10, markeredgewidth=2.5, label='Thinking', linewidth=0),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='#2C3E50',
           markersize=10, markeredgewidth=2.5, label='Mars (RTX A5000)', linewidth=0),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='#E67E22',
           markersize=10, markeredgewidth=2.5, label='RunPod (H100)', linewidth=0),
    Line2D([0], [0], color='gray', linewidth=2, linestyle='--',
           label=f'Trend (R²={r_value**2:.3f})')
]

ax.legend(handles=legend_elements, loc='lower right', fontsize=11,
          framealpha=0.95, edgecolor='black', title_fontsize=12)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'codegen_pass1_energy_tradeoff_labeled.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n📊 Enhanced scatter plot saved!")
print(f"   Trend line: R² = {r_value**2:.3f}, p-value = {p_value:.4f}")
if p_value < 0.05:
    print("   ✅ Statistically significant trend!")
else:
    print("   ⚠️  No statistically significant trend")

In [ ]:
# Filter 4B models for fair comparison (both hardware have 4B)
df_4b = df_clean[df_clean['model_size'] == '4B'].copy()

# Compare by hardware
hardware_comp = df_4b.groupby('hardware')[['pass_rate_pct', 'energy_consumed_kwh', 'duration_hours']].mean()

print("📊 4B Models: Hardware Comparison\n")
print(hardware_comp.round(2))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Performance comparison
hardware_comp['pass_rate_pct'].plot(kind='bar', ax=axes[0], rot=0, color=['#9B59B6', '#2ECC71'])
axes[0].set_title('Pass@1: Mars vs RunPod (4B Models)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Hardware', fontsize=12)
axes[0].set_ylabel('Pass@1 (%)', fontsize=12)
axes[0].set_ylim(85, 100)
axes[0].grid(True, alpha=0.3)

# Energy & Runtime comparison
ax2 = axes[1]
x = np.arange(len(hardware_comp))
width = 0.35

ax2.bar(x - width/2, hardware_comp['energy_consumed_kwh'], width, label='Energy (kWh)', color='skyblue')
ax2.bar(x + width/2, hardware_comp['duration_hours'], width, label='Duration (hours)', color='coral')

ax2.set_title('Energy & Runtime: Mars vs RunPod (4B Models)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Hardware', fontsize=12)
ax2.set_ylabel('Value', fontsize=12)
ax2.set_xticks(x)
ax2.set_xticklabels(hardware_comp.index, rotation=0)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'codegen_hardware_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Energy Efficiency Analysis

### 7.1 Energy Consumption Breakdown

In [ ]:
# Energy analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Energy by model size
energy_by_size = df_clean.groupby('model_size')['energy_consumed_kwh'].mean()
energy_by_size.plot(kind='bar', ax=axes[0, 0], rot=0, color=['skyblue', 'coral'])
axes[0, 0].set_title('Average Energy by Model Size', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Energy (kWh)')
axes[0, 0].grid(True, alpha=0.3)

# 2. Energy by model type
energy_by_type = df_clean.groupby('model_type')['energy_consumed_kwh'].mean()
energy_by_type.plot(kind='bar', ax=axes[0, 1], rot=0, color=['#FF6B6B', '#4ECDC4'])
axes[0, 1].set_title('Average Energy by Model Type', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Energy (kWh)')
axes[0, 1].grid(True, alpha=0.3)

# 3. Component breakdown
energy_components = df_clean[['cpu_energy_kwh', 'gpu_energy_kwh', 'ram_energy_kwh']].mean()
energy_components.plot(kind='pie', ax=axes[1, 0], autopct='%1.1f%%', startangle=90, 
                       colors=['#3498DB', '#E74C3C', '#2ECC71'])
axes[1, 0].set_title('Energy Distribution by Component', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('')

# 4. Emissions by hardware
emissions_by_hw = df_clean.groupby('hardware')['emissions_kg_codecarbon'].mean()
emissions_by_hw.plot(kind='bar', ax=axes[1, 1], rot=0, color=['#9B59B6', '#2ECC71'])
axes[1, 1].set_title('Average CO2 Emissions by Hardware', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Emissions (kg CO2)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'codegen_energy_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Energy Component Breakdown (Average):")
print(f"CPU Energy: {energy_components['cpu_energy_kwh']:.4f} kWh ({energy_components['cpu_energy_kwh']/energy_components.sum()*100:.1f}%)")
print(f"GPU Energy: {energy_components['gpu_energy_kwh']:.4f} kWh ({energy_components['gpu_energy_kwh']/energy_components.sum()*100:.1f}%)")
print(f"RAM Energy: {energy_components['ram_energy_kwh']:.4f} kWh ({energy_components['ram_energy_kwh']/energy_components.sum()*100:.1f}%)")

### 7.2 Performance vs Energy Tradeoff

In [ ]:
# Scatter plot: Pass@1 vs Energy
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

for model_size in df_clean['model_size'].unique():
    for model_type in df_clean['model_type'].unique():
        subset = df_clean[(df_clean['model_size'] == model_size) & (df_clean['model_type'] == model_type)]
        marker = 'o' if model_type == 'Instruct' else '^'
        ax.scatter(
            subset['energy_consumed_kwh'],
            subset['pass_rate_pct'],
            s=150,
            alpha=0.7,
            marker=marker,
            label=f"{model_size} {model_type}"
        )

ax.set_xlabel('Energy Consumed (kWh)', fontsize=12)
ax.set_ylabel('Pass@1 (%)', fontsize=12)
ax.set_title('Pass@1 vs Energy Tradeoff', fontsize=14, fontweight='bold')
ax.legend(title='Configuration', loc='best')
ax.grid(True, alpha=0.3)
ax.set_ylim(85, 102)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'codegen_performance_energy_tradeoff.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate efficiency (Pass@1 per kWh)
df_clean['efficiency'] = df_clean['pass_at_1'] * 100 / df_clean['energy_consumed_kwh']
print("\n📊 Top 5 Most Efficient Configurations (Pass@1 per kWh):")
print(df_clean.nlargest(5, 'efficiency')[['experiment_id', 'pass_rate_pct', 'energy_consumed_kwh', 'efficiency']].to_string(index=False))

## 8. RQ4: Code Generation vs Vulnerability Detection

In [ ]:
# Load vulnerability detection data for comparison
df_vuln = pd.read_csv(VULN_DATA)

print("📊 Task Comparison: Code Generation vs Vulnerability Detection\n")

# Compare average metrics
comparison = pd.DataFrame({
    'Task': ['Code Generation', 'Vulnerability Detection'],
    'Avg Performance': [df_clean['pass_at_1'].mean() * 100, df_vuln['Accuracy'].mean() * 100],
    'Avg Energy (kWh)': [df_clean['energy_consumed_kwh'].mean(), df_vuln['energy_consumed_kwh'].mean()],
    'Avg Duration (hrs)': [df_clean['duration_hours'].mean(), df_vuln['duration_seconds'].mean() / 3600],
    'Avg Emissions (kg CO2)': [df_clean['emissions_kg_codecarbon'].mean(), df_vuln['emissions_kg_codecarbon'].mean()]
})

print(comparison.round(2))

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Performance comparison
comparison.plot(x='Task', y='Avg Performance', kind='bar', ax=axes[0], rot=0, legend=False, color=['#3498DB', '#E74C3C'])
axes[0].set_title('Average Performance: Code Gen vs Vuln Detection', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Performance (%)')
axes[0].set_xlabel('')
axes[0].grid(True, alpha=0.3)
for i, v in enumerate(comparison['Avg Performance']):
    axes[0].text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')

# Energy comparison
comparison.plot(x='Task', y='Avg Energy (kWh)', kind='bar', ax=axes[1], rot=0, legend=False, color=['#3498DB', '#E74C3C'])
axes[1].set_title('Average Energy: Code Gen vs Vuln Detection', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Energy (kWh)')
axes[1].set_xlabel('')
axes[1].grid(True, alpha=0.3)

# Duration comparison
comparison.plot(x='Task', y='Avg Duration (hrs)', kind='bar', ax=axes[2], rot=0, legend=False, color=['#3498DB', '#E74C3C'])
axes[2].set_title('Average Duration: Code Gen vs Vuln Detection', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Duration (hours)')
axes[2].set_xlabel('')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'codegen_vs_vuln_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# Performance improvement
perf_improvement = ((comparison.loc[0, 'Avg Performance'] - comparison.loc[1, 'Avg Performance']) / 
                    comparison.loc[1, 'Avg Performance'] * 100)
energy_reduction = ((comparison.loc[1, 'Avg Energy (kWh)'] - comparison.loc[0, 'Avg Energy (kWh)']) / 
                    comparison.loc[1, 'Avg Energy (kWh)'] * 100)

print(f"\n📊 Key Findings:")
print(f"   Code generation performance is {perf_improvement:.1f}% higher than vulnerability detection")
print(f"   Code generation uses {energy_reduction:.1f}% less energy than vulnerability detection")

## 9. Summary Tables & Export

In [ ]:
# Create comprehensive summary table
summary_table = df_clean[[
    'experiment_id', 'phase', 'hardware', 'model_size', 'model_type', 'prompting',
    'pass_at_1', 'pass_rate_pct', 'passed_samples', 'failed_samples', 'total_samples',
    'energy_consumed_kwh', 'emissions_kg_codecarbon', 'duration_hours',
    'cpu_energy_kwh', 'gpu_energy_kwh', 'ram_energy_kwh'
]].copy()

# Round numeric columns
numeric_cols = summary_table.select_dtypes(include=[np.number]).columns
summary_table[numeric_cols] = summary_table[numeric_cols].round(3)

# Export to Excel
excel_file = OUTPUT_DIR / 'code_generation_comprehensive_analysis.xlsx'
with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    summary_table.to_excel(writer, sheet_name='All Experiments', index=False)
    
    # Add aggregated views
    df_clean.groupby('model_size')[['pass_rate_pct', 'energy_consumed_kwh']].mean().round(2).to_excel(
        writer, sheet_name='By Model Size'
    )
    
    df_clean.groupby('model_type')[['pass_rate_pct', 'energy_consumed_kwh']].mean().round(2).to_excel(
        writer, sheet_name='By Model Type'
    )
    
    df_clean.groupby('prompting')[['pass_rate_pct', 'energy_consumed_kwh']].mean().round(2).to_excel(
        writer, sheet_name='By Prompting'
    )
    
    df_clean.groupby('hardware')[['pass_rate_pct', 'energy_consumed_kwh']].mean().round(2).to_excel(
        writer, sheet_name='By Hardware'
    )

print(f"\n✅ Analysis complete!")
print(f"📊 Excel file saved: {excel_file}")
print(f"🖼️  Visualizations saved to: {OUTPUT_DIR}/")

# Display final summary
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)
print(f"\nTotal Experiments Analyzed: {len(df_clean)}")
print(f"\nBest Pass@1: {df_clean['pass_rate_pct'].max():.2f}% ({df_clean.loc[df_clean['pass_rate_pct'].idxmax(), 'experiment_id']})")  
print(f"Most Energy Efficient: {df_clean.loc[df_clean['efficiency'].idxmax(), 'experiment_id']}")
print(f"\nAverage Pass@1: {df_clean['pass_at_1'].mean()*100:.2f}%")
print(f"Average Energy: {df_clean['energy_consumed_kwh'].mean():.3f} kWh")
print(f"Average Emissions: {df_clean['emissions_kg_codecarbon'].mean():.3f} kg CO2")
print("\n" + "="*80)

## 10. Key Findings

**To be filled after analysis:**

### RQ1: Model Size & Reasoning Impact
- Model size impact on Pass@1: [TBD]
- Reasoning capability impact: [TBD]

### RQ2: Prompting Strategy
- Zero-shot vs Few-shot: [TBD]

### RQ3: Hardware Impact
- Performance comparison: [TBD]
- Speed comparison: [TBD]

### RQ4: Task Comparison
- Code gen vs vuln detection performance: [TBD]
- Energy efficiency differences: [TBD]

### Overall Findings
- Best configuration: [TBD]
- Energy-optimal setup: [TBD]